In [3]:
# Path to the credentials file you downloaded from Aura
CREDS_PATH = "/slurm/homes/pchandra/xai-knowledge-graph/NE04J_password.txt"   # adjust to wherever you saved it

creds = {}
with open(CREDS_PATH) as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            creds[key.strip()] = val.strip()

# Sanity check (without printing the password)
print("Keys found:", list(creds.keys()))
print("URI       :", creds.get("NEO4J_URI"))
print("Username  :", creds.get("NEO4J_USERNAME"))
print("Pwd length:", len(creds.get("NEO4J_PASSWORD", "")))   # should be ~25-32 chars

Keys found: ['NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD', 'NEO4J_DATABASE', 'AURA_INSTANCEID', 'AURA_INSTANCENAME']
URI       : neo4j+s://9f8cae61.databases.neo4j.io
Username  : 9f8cae61
Pwd length: 43


In [4]:
from neo4j import GraphDatabase

def try_connect(user, pw, label):
    try:
        with GraphDatabase.driver(creds["NEO4J_URI"], auth=(user, pw)) as driver:
            driver.verify_connectivity()
            with driver.session() as session:
                msg = session.run("RETURN 'hello from jupyter' AS m").single()["m"]
                print(f"✓ {label} — {msg}")
                return True
    except Exception as e:
        print(f"✗ {label} — {type(e).__name__}: {str(e)[:100]}")
        return False

# Try with the standard Aura default username first
try_connect("neo4j", creds["NEO4J_PASSWORD"], "username='neo4j'")

# Then try with whatever was parsed from the file
try_connect(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"], f"username='{creds['NEO4J_USERNAME']}'")

✗ username='neo4j' — AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to auth
✓ username='9f8cae61' — hello from jupyter


True